# Consistency: a label-free judge from corrupted reasoning (offline test): steps 3 and 4

Good reasoning tends to give good solutions, broken reasoning bad ones. The SFT policy samples normal completions; the
reasoning of each is then corrupted, and the policy writes only the solution + answer from it, with just enough room
for a solution:
- **empty**: no reasoning at all
- **swap**: the reasoning of another puzzle (same size)
- **cut**: the first `CUT_FRACTION` of the trace's own reasoning (whole lines), so the cut scales with each trace


DPO judge: preferred = the normal solution, rejected = same puzzle solution from
corrupted reasoning. 

The notebook uses data produced by Consistency_Corrupt_Judge_Generate and the adapter produced by Consistency_Countdown.

In [1]:
# [config] Everything you choose lives here.
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
SFT_RUN = "Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500"
# the GRPO start (trace v4 without unsolvable traces, small SFT, 2026-10-26; v4 with them: -imp38x38); the 2026-10-23/24 offline test used the mixed SFT, OLD_SFT_RUN
OLD_SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"
SOLVER_DPO_RUN = "judge_dpo_swap30extra30slip20miss20_n2250_n3x2250_b0.1_lr2e-05_ep1"     # the solver-pair judge (comparison;
                                         # trained by Consistency_Countdown's [judge DPO] under SFT_DIR)
DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
SFT_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
ADAPTER_PATH = f"{SFT_DIR}/adapter"                          # the policy that samples, and the DPO start and reference
SOLVER_JUDGE_PATH = f"{SFT_DIR}/{SOLVER_DPO_RUN}/adapter"   # "solver": the solver-pair judge of THIS SFT (2026-10-26)
SOLVER_OLD_JUDGE_PATH = f"{DRIVE}/countdown/sft/{OLD_SFT_RUN}/{SOLVER_DPO_RUN}/adapter"   # "solver_old": the mixed SFT's,
# which scored the GRPO dumps (their logged judge_mean: [dump probe]'s check); differs from the corrupt judge in pair
# type AND SFT start
# Step 1: normal completions, one per puzzle, as GRPO samples them. Puzzles: a new seeded draw (SEED + 6) without every
# puzzle of the test, SFT, GRPO RL and GRPO 3-number draws (Consistency_Countdown's real-pair exclusion) and without the
# dump puzzles; the last VAL_SHARE of each size give the validation pairs
N_PUZZLES_4, N_PUZZLES_3 = 1000, 1000
VAL_SHARE = 0.1
TEMPERATURE = 0.9
MAX_TOKENS = 1024
SEED = 0
# Step 2: corruptions of each normal completion's reasoning; the policy then writes <solution> and <answer> only
KINDS = ("empty", "swap", "cut")
HOT_TEMPERATURES = (1.3, 1.5)            # hot: a new reasoning at each temperature (kind "hot1.3", ...), own file
CUT_FRACTION = 0.5                       # cut: keep this share of the reasoning's tokens, back to the last whole line
SOL_MAX_TOKENS = None                    # room for solution + answer; None = the 99th percentile over the normal
                                         # completions (from "<solution>" through "</answer>"), the same cap for both
PLAUSIBILITY_MAX_SOLUTION_TOKENS = 128   # as the GRPO reward: longer solutions, or no answer, are not scored
VLLM_GPU_MEMORY_UTILIZATION = 0.5
# Step 3: DPO as the current judge ([judge DPO] of Consistency_Countdown): continue the SFT adapter, reference = the SFT
# adapter, batch 16
DPO_BETA, DPO_LR, DPO_EPOCHS = 0.1, 2e-5, 1
TRAIN_KINDS = ("empty", "cut")          # the corruptions whose pairs train the judge, e.g. ("hot1.3",) (validation:
                                         # every kind, hot ones included)
RETRAIN_DPO = False
# Step 4: the GRPO dumps (as Consistency_Judge_Score_Probe): the mean-34 run from the old mixed SFT (the new SFT has no
# GRPO runs yet), so the judge is tested on another policy's fakes; also excluded from the step-1 puzzles
GRPO_DIR = f"{DRIVE}/countdown/grpo/{OLD_SFT_RUN}-judge_dpo-start_sft"
RUN = f"{GRPO_DIR}/runs/grpo-mean-missmed-mix3_0.5-expr-kl0.02-swap30extra30slip20miss20_n2250_n3x2250-mean-34"
DUMP_DIRS = [(RUN, 0, 99), (RUN + "-cont100b", 100, 10**6)]   # (run, first step, last step)
CHUNK = 8                                # sequences per scoring forward pass
UNASSIGN_WHEN_DONE = True                # release the Colab runtime after the last cell (stops the compute bill)

OUT_DIR = (f"{SFT_DIR}/judge_corrupt_{'-'.join(KINDS)}" + (f"_cut{CUT_FRACTION:g}" if "cut" in KINDS else "")
           + f"_n4x{N_PUZZLES_4}_n3x{N_PUZZLES_3}_t{TEMPERATURE:g}_seed{SEED}")
NORMAL_PATH, CORRUPT_PATH = f"{OUT_DIR}/normal.csv", f"{OUT_DIR}/corrupt.csv"
HOT_PATHS = {t: f"{OUT_DIR}/corrupt_hot{t:g}.csv" for t in HOT_TEMPERATURES}
ALL_KINDS = (*KINDS, *(f"hot{t:g}" for t in HOT_TEMPERATURES))
_dpo = f"b{DPO_BETA:g}_lr{DPO_LR:g}_ep{DPO_EPOCHS}"
JUDGE_DIR = f"{OUT_DIR}/dpo_" + ("" if set(TRAIN_KINDS) == set(KINDS) else f"{'-'.join(TRAIN_KINDS)}_") + _dpo
ALL_KINDS_JUDGE_PATH = f"{OUT_DIR}/dpo_{_dpo}/adapter"   # trained on every kind in KINDS (a comparison, if it exists)
JUDGE_PATH = f"{JUDGE_DIR}/adapter"
assert set(KINDS) <= {"empty", "swap", "cut"} and set(TRAIN_KINDS) <= set(ALL_KINDS), (KINDS, TRAIN_KINDS)
print(f"OUT_DIR={OUT_DIR}\nJUDGE_DIR={JUDGE_DIR}")

OUT_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0
JUDGE_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1


In [2]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN; Llama is gated). Re-running is harmless.
import os
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # read by transformers and vLLM; no huggingface_hub import before [install]
for f in (f"{ADAPTER_PATH}/adapter_model.safetensors", f"{SOLVER_JUDGE_PATH}/adapter_model.safetensors",
          f"{SOLVER_OLD_JUDGE_PATH}/adapter_model.safetensors",
          f"{CODE_DIR}/countdown.py", f"{CODE_DIR}/prompts.py", f"{CODE_DIR}/rewards.py", f"{CODE_DIR}/judge_data.py"):
    assert os.path.exists(f), f"missing {f}"

Mounted at /content/drive


In [3]:
# [install] TRL and PEFT for DPO and scoring (no vLLM: nothing samples here), on Colab's torch and transformers; no
# restart (nothing they replace is imported yet). Skipped when trl and peft are installed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("trl") and importlib.util.find_spec("peft"):
    print("[install] skipped: trl and peft are installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):   # removed either way: Colab ships 0.10, peft refuses to import below 0.16
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

[install] done
[install] removed torchao


In [4]:
# [setup] Tokenizer, the Countdown code (Drive), the GRPO dumps (step 4, and excluded from the puzzles), the puzzles, and
# the helpers: the judge's view of a completion (as the GRPO reward scores it) and what kind of answer it is.
import os, sys, re, glob, json, random, importlib, math, gc
from collections import Counter
from fractions import Fraction
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS)
tok.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tok.chat_template)   # pinned date

def chat_ids(messages):
    """Templated prompt + the assistant header as token ids, no extra BOS."""
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
               add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import prompts, countdown, traces, rewards, judge_data
for _m in (prompts, countdown, traces, rewards, judge_data):   # prompts first: the others import from it
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, bare_question, judge_messages, NEUTRAL_SYSTEM_PROMPT
from rewards import tag, diagnose
assert judge_messages([1, 2, 3, 4], 10) == [{"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
                                            {"role": "user", "content": bare_question(countdown_prompt([1, 2, 3, 4], 10))}], \
    "judge prompt differs from the GRPO reward's"

_NUM = r"\(?\s*(-?\d+(?:\.\d+)?(?:\s*/\s*\d+)?)\s*\)?"      # Consistency_Countdown's check + decimals: (-5), 3/4, 6.5
_STEP = re.compile(rf"^\s*{_NUM}\s*([+\-*/×÷x])\s*{_NUM}\s*=\s*{_NUM}\s*\.?\s*$")

def replay(sol, numbers):
    """Replays the solution's steps from the given numbers: "false" (a step's result is wrong), "unavailable" (every
    step true, an operand is neither a given number nor an earlier result, or is used twice), "valid" (and the last
    result), or "unparsed" (a line that is not "a op b = c", after dropping a trailing "Left: ...")."""
    avail = Counter(Fraction(x) for x in numbers)
    bad = False
    lines = [l for l in (sol or "").splitlines() if l.strip()]
    if not lines:
        return "unparsed"
    for line in lines:
        m = _STEP.match(line.split("Left:")[0])
        if not m:
            return "unparsed"
        a, op, b, c = m.groups()
        a, b, c = (Fraction(s.replace(" ", "")) for s in (a, b, c))
        r = {"+": a + b, "-": a - b, "*": a * b, "×": a * b, "x": a * b}.get(op, a / b if b else None)
        if r != c:
            return "false"
        for v in (a, b):
            bad |= avail[v] < 1
            avail[v] -= 1
        avail[c] += 1
    return "unavailable" if bad else ("valid", c)

def klass(sol, ans, numbers, target):
    """correct / fake (a false step in the solution) / wrong numbers (every step true, a number made up or reused) /
    honest miss (a valid route from the given numbers that misses the target) / bad answer (a valid route to the
    target, the answer gets it wrong) / other (a solution line we can't parse) / no answer."""
    if not isinstance(ans, str) or not ans.strip():
        return "no answer"
    if check_answer(ans, list(numbers), int(target)):
        return "correct"
    r = replay(sol, numbers)
    if isinstance(r, tuple):
        return "bad answer" if r[1] == target else "honest miss"
    return {"false": "fake", "unavailable": "wrong numbers", "unparsed": "other"}[r]

def scorable(sol, ans):
    """Scored by the GRPO reward: a solution of at most PLAUSIBILITY_MAX_SOLUTION_TOKENS tokens and an answer."""
    return (isinstance(sol, str) and bool(sol.strip()) and isinstance(ans, str) and bool(ans.strip())
            and len(tok(sol, add_special_tokens=False)["input_ids"]) <= PLAUSIBILITY_MAX_SOLUTION_TOKENS)

judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"

# The GRPO dumps (Consistency_Judge_Score_Probe's [data]): step 4's test set
rows = []
for run, lo, hi in DUMP_DIRS:
    for f in sorted(glob.glob(f"{run}/step_*.txt")):
        step = int(re.search(r"step_(\d+)", f).group(1))
        if not lo <= step <= hi:
            continue
        for blk in open(f).read().split("### Sample ")[1:]:
            sc = re.search(r"puzzle=(\d+) correct=(\w+) judge_mean=(\S+)", blk)
            prompt = blk.split("[Prompt]\n", 1)[1].split("\n\n[Response]\n", 1)[0]
            resp = blk.split("[Response]\n", 1)[1].split("\n\n" + "=" * 20, 1)[0]
            pm = re.match(r"Using the numbers (.*?), create an expression that equals (\d+)\.", prompt)
            nums, tgt = [int(x) for x in pm.group(1).split(", ")], int(pm.group(2))
            assert countdown_prompt(nums, tgt)[-1]["content"] == prompt, f"prompt differs: {f}"
            sol, ans = tag(resp, "solution"), tag(resp, "answer")
            rows.append({"step": step, "pid": int(sc.group(1)), "n": len(nums), "numbers": nums, "target": tgt,
                         "correct": sc.group(2) == "True", "judge_logged": float(sc.group(3)),
                         "solution": sol, "answer": ans, "kind": klass(sol, ans, nums, tgt)})
dumps = pd.DataFrame(rows)
dumps["group"] = dumps.step.astype(str) + "/" + dumps.pid.astype(str)
assert len(dumps) and (dumps.groupby("group").size() == 16).all(), "no dumps, or a group without 16 completions"
_dis = ((dumps.kind == "correct") != dumps.correct).mean()
assert _dis < 0.01, f"klass and the logged checker disagree on {_dis:.1%} of the dumps"
print(f"dumps: {len(dumps)} completions, {dumps.group.nunique()} groups, steps {sorted(dumps.step.unique())}")

# Puzzles: Consistency_Countdown's real-pair exclusion (test draw, SFT draw, GRPO RL and 3-number draws at the GRPO
# defaults), a new seed, and none of the dump puzzles
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))
_no = {_key(_as(r)) for seed, size in ((SEED, 20 * 200), (SEED + 1, 6 * (4000 + 100)), (SEED + 2, 4 * (4000 + 100)),
                                        (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)}
_no |= {_key(r) for r in dumps[["numbers", "target"]].to_dict("records")}
_want = {4: N_PUZZLES_4, 3: N_PUZZLES_3}
_got = {4: [], 3: []}
for r in _seeded(SEED + 6, 10 * (N_PUZZLES_4 + N_PUZZLES_3)):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_got[n]) < _want[n]:
        _no.add(_key(p))
        _got[n].append(p)
assert all(len(_got[n]) == _want[n] for n in _want), {n: len(v) for n, v in _got.items()}
puzzles = []
for n in (4, 3):
    n_val = round(VAL_SHARE * _want[n])
    puzzles += [{**p, "pid": len(puzzles) + i, "n": n, "val": i >= _want[n] - n_val} for i, p in enumerate(_got[n])]
print(f"{len(puzzles)} puzzles (4-number {N_PUZZLES_4}, 3-number {N_PUZZLES_3}; validation "
      f"{sum(p['val'] for p in puzzles)}), disjoint from the test, SFT, GRPO and dump puzzles")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

dumps: 2560 completions, 160 groups, steps [np.int64(0), np.int64(20), np.int64(40), np.int64(60), np.int64(80), np.int64(100), np.int64(120), np.int64(140), np.int64(160), np.int64(180), np.int64(200), np.int64(220), np.int64(240), np.int64(260), np.int64(280), np.int64(300), np.int64(320), np.int64(340), np.int64(360), np.int64(380)]


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.85MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

2000 puzzles (4-number 1000, 3-number 1000; validation 200), disjoint from the test, SFT, GRPO and dump puzzles


In [5]:
# [load] Steps 1 and 2 from Consistency_Corrupt_Judge_Generate: the normal and the corrupted completions (KINDS, then
# the hot ones saved so far; TRAIN_KINDS must be among them).
assert os.path.exists(NORMAL_PATH) and os.path.exists(CORRUPT_PATH), "run Consistency_Corrupt_Judge_Generate first"
_hot = [p for p in HOT_PATHS.values() if os.path.exists(p)]
_read = lambda p: pd.read_csv(p, keep_default_na=False, na_values=[""])   # the answer "None" stays a string
normal = _read(NORMAL_PATH)
corrupt = pd.concat([_read(p) for p in (CORRUPT_PATH, *_hot)], ignore_index=True)
assert set(TRAIN_KINDS) <= set(corrupt.kind), f"no pairs for {set(TRAIN_KINDS) - set(corrupt.kind)}: run Consistency_Corrupt_Judge_Generate"
SOL_MAX_TOKENS = int(corrupt.budget.iloc[0])
print(f"[load] loaded {NORMAL_PATH}, {CORRUPT_PATH} and {len(_hot)} of {len(HOT_PATHS)} hot files: kinds "
      f"{corrupt.kind.value_counts().to_dict()} (room for the solution: {SOL_MAX_TOKENS} tokens)")

[load] loaded /content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/normal.csv, /content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/corrupt.csv and 2 of 2 hot files: kinds {'empty': 1328, 'swap': 1328, 'cut': 1328, 'hot1.3': 1328, 'hot1.5': 1328} (room for the solution: 80 tokens)


In [6]:
# [label gap] Step 2 check (the checker measures, nothing trains on it): per size, what the normal and the corrupted
# reasoning give: usable share (finished, solution scored by the GRPO reward), accuracy and kinds of answer among the
# usable ones. Then per pair (normal vs corrupted, same puzzle, both usable): the label is right when the normal is
# correct and the corrupted wrong, inverted in the opposite case, and says nothing about correctness when both agree
# (both wrong: is the corrupted one more often a fake?). Stop here if right is not well above inverted.
CLASSES = ["correct", "fake", "wrong numbers", "honest miss", "bad answer", "other"]
both = pd.concat([normal.assign(kind="normal"), corrupt], ignore_index=True)
rows = []
for (n, kind), d in both.groupby(["n", "kind"], sort=False):
    u = d[d.usable]
    rows.append({"n": n, "source": kind, "rows": len(d), "usable": d.usable.mean(), "accuracy": (u["class"] == "correct").mean(),
                 **{c: (u["class"] == c).mean() for c in CLASSES[1:]},
                 "reasoning kept": (d.reasoning_tokens / d.of_tokens).mean() if kind == "cut" else np.nan})
gap = pd.DataFrame(rows).set_index(["n", "source"]).sort_index()
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("by source (accuracy and kinds among the usable rows):\n" + gap.to_string())

_nu = normal[normal.usable].set_index("pid")
pairs_all = corrupt[corrupt.usable].join(_nu[["solution", "answer", "class"]], on="pid", rsuffix="_normal")
pairs_all["identical"] = [judge_text(a, b) == judge_text(c, d) for a, b, c, d in
                          zip(pairs_all.solution, pairs_all.answer, pairs_all.solution_normal, pairs_all.answer_normal)]
rows = []
for (n, kind), d in pairs_all.groupby(["n", "kind"], sort=False):
    nc, cc = d.class_normal == "correct", d["class"] == "correct"
    bw = d[~nc & ~cc]
    rows.append({"n": n, "kind": kind, "pairs": len(d), "identical": d.identical.mean(), "right": (nc & ~cc).mean(),
                 "inverted": (~nc & cc).mean(), "both correct": (nc & cc).mean(), "both wrong": len(bw) / len(d),
                 "both wrong: fake normal": (bw.class_normal == "fake").mean(),
                 "both wrong: fake corrupted": (bw["class"] == "fake").mean()})
lab = pd.DataFrame(rows).set_index(["n", "kind"]).sort_index()
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("\npairs (normal vs corrupted, same puzzle, both usable):\n" + lab.to_string())
gap.to_csv(f"{OUT_DIR}/label_gap_sources.csv")
lab.to_csv(f"{OUT_DIR}/label_gap_pairs.csv")

by source (accuracy and kinds among the usable rows):
          rows  usable  accuracy  fake  wrong numbers  honest miss  bad answer  other  reasoning kept
n source                                                                                             
3 cut      821   0.976     0.029 0.683          0.196        0.079       0.002  0.011           0.465
  empty    821   0.976     0.210 0.366          0.408        0.007       0.001  0.007             NaN
  hot1.3   821   0.714     0.613 0.244          0.108        0.009       0.002  0.026             NaN
  hot1.5   821   0.295     0.645 0.285          0.050        0.008       0.008  0.004             NaN
  normal  1000   0.821     0.597 0.177          0.213        0.006       0.000  0.007             NaN
  swap     821   0.998     0.000 0.189          0.801        0.000       0.000  0.010             NaN
4 cut      507   0.970     0.083 0.593          0.124        0.167       0.014  0.018           0.469
  empty    507   0.846     0

In [7]:
# [judge DPO] Step 3: chosen = the normal solution + answer, rejected = the one from corrupted reasoning (same puzzle;
# both usable, texts not identical), in the judge's format (neutral system prompt + bare puzzle). Training pairs from
# the training puzzles and TRAIN_KINDS, validation pairs from the last VAL_SHARE (every kind). Consistency_Countdown's [judge DPO] otherwise:
# continue the SFT adapter, reference = the SFT adapter (TRL's frozen "ref" copy), batch 16, cosine, 5% warmup.
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

def to_dpo(d):
    return [{"prompt": judge_messages(json.loads(r.numbers), int(r.target)),
             "chosen": [{"role": "assistant", "content": judge_text(r.solution_normal, r.answer_normal)}],
             "rejected": [{"role": "assistant", "content": judge_text(r.solution, r.answer)}],
             "kind": r.kind, "n": int(r.n), "correct_normal": r.class_normal == "correct",
             "correct_rejected": r["class"] == "correct"} for _, r in d.iterrows()]

_p = pairs_all[~pairs_all.identical]
train_pairs = to_dpo(_p[~_p.val & _p.kind.isin(TRAIN_KINDS)].sample(frac=1, random_state=SEED))   # interleaved
val_pairs = to_dpo(_p[_p.val])
print(f"{len(train_pairs)} training pairs {pd.Series([(x['n'], x['kind']) for x in train_pairs]).value_counts().sort_index().to_dict()}, "
      f"{len(val_pairs)} validation pairs; e.g. ({train_pairs[0]['kind']})\nCHOSEN\n{train_pairs[0]['chosen'][0]['content']}"
      f"\nREJECTED\n{train_pairs[0]['rejected'][0]['content']}")
if os.path.exists(f"{JUDGE_PATH}/adapter_config.json") and not RETRAIN_DPO:
    print(f"[judge DPO] skipped: {JUDGE_PATH} exists (RETRAIN_DPO = False)")
else:
    from trl import DPOConfig, DPOTrainer
    _cols = ("prompt", "chosen", "rejected")
    _base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
    _base.config.pad_token_id = tok.pad_token_id
    dpo_model = PeftModel.from_pretrained(_base, ADAPTER_PATH, is_trainable=True)
    _steps = -(-len(train_pairs) // 16) * DPO_EPOCHS
    dpo_args = DPOConfig(
        output_dir=f"{JUDGE_DIR}/checkpoints", beta=DPO_BETA, learning_rate=DPO_LR, num_train_epochs=DPO_EPOCHS,
        lr_scheduler_type="cosine", warmup_steps=max(1, round(0.05 * _steps)),
        per_device_train_batch_size=4, gradient_accumulation_steps=4, per_device_eval_batch_size=4,
        max_length=512, bf16=True, gradient_checkpointing=True, logging_steps=10, save_strategy="no",
        eval_strategy="steps", eval_steps=50, report_to="none", seed=SEED)
    dpo_trainer = DPOTrainer(model=dpo_model, args=dpo_args, processing_class=tok,
                             train_dataset=Dataset.from_list([{k: x[k] for k in _cols} for x in train_pairs]),
                             eval_dataset=Dataset.from_list([{k: x[k] for k in _cols} for x in val_pairs]))
    assert set(dpo_model.peft_config) == {"default", "ref"}, f"expected a frozen copy of the SFT adapter as reference: {list(dpo_model.peft_config)}"
    print(f"{len(train_pairs)} pairs, {_steps} optimizer steps; reference = the SFT adapter ('ref')")
    dpo_trainer.train()
    dpo_model.save_pretrained(JUDGE_PATH, selected_adapters=["default"])   # the trained adapter only (not "ref")
    _log = pd.DataFrame(dpo_trainer.state.log_history)
    _log.to_csv(f"{JUDGE_DIR}/train_log.csv", index=False)
    pd.DataFrame(train_pairs + val_pairs).assign(split=["train"] * len(train_pairs) + ["val"] * len(val_pairs),
                                                 chosen=lambda d: d.chosen.str[0].str["content"],
                                                 rejected=lambda d: d.rejected.str[0].str["content"],
                                                 prompt=lambda d: d.prompt.str[1].str["content"]
                                                 ).to_csv(f"{JUDGE_DIR}/dpo_pairs.csv", index=False)
    print("saved", JUDGE_PATH)
    _show = [c for c in ("step", "loss", "eval_loss", "eval_rewards/accuracies", "eval_rewards/margins") if c in _log]
    print(_log[_show].dropna(how="all", subset=_show[1:]).to_string(index=False))
    del dpo_trainer, dpo_model, _base
    gc.collect(); torch.cuda.empty_cache()

2205 training pairs {(3, 'cut'): 710, (3, 'empty'): 701, (4, 'cut'): 411, (4, 'empty'): 383}, 481 validation pairs; e.g. (empty)
CHOSEN
13 * 26 = 338
338 - 61 = 277
277 - 229 = 48
Answer: ((13 * 26) - 61) - 229
REJECTED
61 + 26 = 87
87 - 13 = 48
Answer: (61 + 26) - 13


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

... (3 outputs left out)

Dropping fully truncated examples from eval dataset:   0%|          | 0/481 [00:00<?, ? examples/s]

2205 pairs, 138 optimizer steps; reference = the SFT adapter ('ref')


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.277180,0.473914,0.171968,212407.000000,0.072747,0.190298,0.876227,-1.451000,-2.864459,0.778926,1.413459,-23.586706,-50.511328
100,0.395194,0.473672,0.185398,425590.000000,0.024372,0.155642,0.863229,-1.587196,-3.087549,0.774793,1.500354,-24.948664,-52.742234
138,0.240687,0.454981,0.186594,586860.000000,0.028451,0.158958,0.863702,-1.550195,-3.130092,0.783058,1.579898,-24.578654,-53.167663


saved /content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1/adapter
 step     loss  eval_loss  eval_rewards/accuracies  eval_rewards/margins
   10 0.645823        NaN                      NaN                   NaN
   20 0.430260        NaN                      NaN                   NaN
   30 0.343467        NaN                      NaN                   NaN
   40 0.347700        NaN                      NaN                   NaN
   50 0.277180        NaN                      NaN                   NaN
   50      NaN   0.473914                 0.778926              1.413459
   60 0.366706        NaN                      NaN                   NaN
   70 0.316182        NaN                      NaN                   NaN
   80 0.276406        NaN                      NaN                   NaN
   90 0.276

In [8]:
# [validation] The judges, one base model, an adapter each (+ adapter off): "base" (untrained), "sft" (the corrupted
# judge's start and DPO reference), "solver" (solver-made pairs, this SFT), "solver_old" (the same recipe on the mixed SFT), "corrupt" (step 3, TRAIN_KINDS),
# "corrupt_all" (trained on every kind; when TRAIN_KINDS is fewer and that judge exists). Score = mean log-prob of
# solution + answer + EOS under the judge prompt (the GRPO reward). On the validation pairs: share with chosen >
# rejected, per corruption kind, and per label (right: the normal correct and the corrupted wrong; inverted; both
# correct; both wrong): a judge that learned only "corrupted or not" is as high on both wrong as on right.
# Then correct vs stop (2026-10-24): the trace-v4 policy stops ("No solution found." / "none") where its search fails,
# and most training pairs are stop > fake; the judge must still put a correct answer above a stop, or GRPO with it
# rewards stopping. Per validation puzzle (usable normal completion), the stop text the policy writes vs the gold
# solution (judge_data.gold, every puzzle) and vs the policy's own correct one; and the stop vs its own fake (stop
# should win). Share with the first above the second; saved to JUDGE_DIR/validation_stop.csv.
import contextlib
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel
assert os.path.exists(f"{JUDGE_PATH}/adapter_config.json"), "run [judge DPO] first"
JUDGES = {"base": None, "sft": ADAPTER_PATH, "solver": SOLVER_JUDGE_PATH, "solver_old": SOLVER_OLD_JUDGE_PATH,
          "corrupt": JUDGE_PATH}
if JUDGE_PATH != ALL_KINDS_JUDGE_PATH and os.path.exists(f"{ALL_KINDS_JUDGE_PATH}/adapter_config.json"):
    JUDGES["corrupt_all"] = ALL_KINDS_JUDGE_PATH
judge_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda"),
                                        ADAPTER_PATH, adapter_name="sft").eval()
for _name, _path in JUDGES.items():
    if _path and _name != "sft":
        judge_model.load_adapter(_path, adapter_name=_name)

def score(items, judge):
    """Mean log-prob of each (numbers, target, text) under one judge; similar lengths per forward pass."""
    seqs = [(chat_ids(judge_messages(nums, tgt)), tok(text, add_special_tokens=False)["input_ids"] + [tok.eos_token_id])
            for nums, tgt, text in items]
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    out = [None] * len(seqs)
    if JUDGES[judge]:
        judge_model.set_adapter(judge)
    with torch.no_grad(), (judge_model.disable_adapter() if JUDGES[judge] is None else contextlib.nullcontext()):
        for s in range(0, len(order), CHUNK):
            chunk = order[s:s + CHUNK]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in chunk)
            ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(chunk):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            logits = judge_model(input_ids=ids.cuda(), attention_mask=attn.cuda()).logits
            for j, i in enumerate(chunk):
                p, t = seqs[i]
                z = logits[j, len(p) - 1:len(p) + len(t) - 1].float()
                out[i] = (z.gather(-1, torch.tensor(t, device=z.device)[:, None])[:, 0] - torch.logsumexp(z, -1)).mean().item()
            del logits
    return out

_vp = pd.DataFrame([{"kind": x["kind"], "n": x["n"],
                     "label": {(True, False): "right", (False, True): "inverted", (True, True): "both correct",
                               (False, False): "both wrong"}[(x["correct_normal"], x["correct_rejected"])]}
                    for x in val_pairs])
_args = lambda part: [(json.loads(r.numbers), int(r.target), t) for r, t in
                      zip(_p[_p.val].itertuples(), (x[part][0]["content"] for x in val_pairs))]
for j in JUDGES:
    c, r = score(_args("chosen"), j), score(_args("rejected"), j)
    _vp[j] = [float(a > b) + 0.5 * float(a == b) for a, b in zip(c, r)]
val_res = pd.concat([_vp.groupby(k)[list(JUDGES)].mean().T.add_prefix(f"{k}: ") for k in ("kind", "label")], axis=1)
val_res.insert(0, "all", _vp[list(JUDGES)].mean())
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print(f"validation pairs ({len(_vp)}; per label: {_vp.label.value_counts().to_dict()}), chosen > rejected:\n" + val_res.to_string())
val_res.to_csv(f"{JUDGE_DIR}/validation.csv")

_STOP = judge_text("No solution found.", "none")
_cs = []
for r in normal[normal.val & normal.usable].to_dict("records"):
    nums, tgt = json.loads(r["numbers"]), int(r["target"])
    g = judge_data.gold(nums, tgt, random.Random(SEED + r["pid"]))
    if g is not None:
        _cs.append({"test": "gold > stop", "nums": nums, "tgt": tgt, "a": judge_data.judge_text(*g), "b": _STOP})
    if r["class"] == "correct":
        _cs.append({"test": "own correct > stop", "nums": nums, "tgt": tgt, "a": judge_text(r["solution"], r["answer"]), "b": _STOP})
    if r["class"] == "fake":
        _cs.append({"test": "stop > own fake", "nums": nums, "tgt": tgt, "a": _STOP, "b": judge_text(r["solution"], r["answer"])})
_cs = pd.DataFrame(_cs)
for j in JUDGES:
    _sa, _sb = (score(list(zip(_cs.nums, _cs.tgt, _cs[k])), j) for k in ("a", "b"))
    _cs[j] = [float(x > y) + 0.5 * float(x == y) for x, y in zip(_sa, _sb)]
stop_res = _cs.groupby("test")[list(JUDGES)].mean()
stop_res.insert(0, "n", _cs.test.value_counts())
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("\ncorrect vs stop (validation puzzles; share with the first above the second):\n" + stop_res.to_string())
stop_res.to_csv(f"{JUDGE_DIR}/validation_stop.csv")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

validation pairs (481; per label: {'both wrong': 213, 'right': 188, 'both correct': 44, 'inverted': 36}), chosen > rejected:
             all  kind: cut  kind: empty  kind: hot1.3  kind: hot1.5  kind: swap  label: both correct  label: both wrong  label: inverted  label: right
base       0.748      0.711        0.672         0.551         0.676       0.977                0.591              0.718            0.417         0.883
sft        0.726      0.736        0.624         0.478         0.559       0.985                0.659              0.671            0.028         0.936
solver     0.713      0.719        0.656         0.420         0.412       0.992                0.523              0.615            0.056         0.995
solver_old 0.759      0.760        0.760         0.449         0.471       0.992                0.523              0.709            0.083         1.000
corrupt    0.825      0.876        0.840         0.536         0.559       0.985                0.682              

In [ ]:
# [dump probe] Step 4, on the 2,560 GRPO dump completions (160 groups of 16; the GRPO policy from step 0 to 400), the
# view the GRPO reward scores (solution of at most 128 tokens + answer; others unscored). Check: the old solver judge
# (solver_old, which scored these GRPO runs) reproduces the logged judge_mean. Per judge: correct > wrong on the same puzzle (pairs) and fully separated groups
# (Consistency_Judge_Score_Probe: solver 0.992 / 0.889, untrained Llama-3B 0.818); mean score per kind of answer; the
# order among wrong answers on the same puzzle (want honest miss and wrong numbers above fake: every step true beats a
# false step that lands on the target); and the share of within-group variance between correct and wrong (raw).
_sc = dumps.apply(lambda r: scorable(r.solution, r.answer), axis=1)
_items = [(r.numbers, r.target, judge_text(r.solution, r.answer)) for r in dumps[_sc].itertuples()]
for j in JUDGES:
    dumps[j] = np.nan
    dumps.loc[_sc, j] = score(_items, j)
_d = (dumps.solver_old - dumps.judge_logged).abs().dropna()
print(f"solver_old judge vs the logged judge_mean: median |diff| {_d.median():.4f}, max {_d.max():.4f} over {len(_d)}")
assert _d.median() < 0.02, "the scoring does not reproduce the GRPO reward's judge score"
dumps.drop(columns=["solution", "answer"]).to_csv(f"{JUDGE_DIR}/dump_scores.csv", index=False)

g = dumps.groupby("group")

def pair_share(a, b, col):
    """Over same-puzzle pairs (x from rows a, y from rows b): share with x > y (ties 1/2), and the number of pairs."""
    w = n = 0
    for _, d in g:
        x, y = d.loc[a(d), col].dropna().values, d.loc[b(d), col].dropna().values
        diff = x[:, None] - y[None, :]
        w += (diff > 0).sum() + 0.5 * (diff == 0).sum()
        n += diff.size
    return (w / n if n else np.nan), n

def measures(col):
    pw, _ = pair_share(lambda d: d.correct, lambda d: ~d.correct, col)
    sep = [d.loc[d.correct, col].min() > d.loc[~d.correct, col].max()
           for _, d in g if d.loc[d.correct, col].notna().any() and d.loc[~d.correct, col].notna().any()]
    between = within = 0.0
    for _, d in g:
        x = d[[col, "correct"]].dropna()
        if len(x) < 2:
            continue
        tot = ((x[col] - x[col].mean()) ** 2).sum()
        w = sum(((s - s.mean()) ** 2).sum() for s in (x.loc[x.correct, col], x.loc[~x.correct, col]) if len(s))
        between, within = between + tot - w, within + w
    k = dumps.groupby("kind")[col].mean()
    return {"correct>wrong": pw, "groups sep.": np.mean(sep),
            **{f"mean: {c}": k.get(c) for c in CLASSES},
            "honest miss>fake": pair_share(lambda d: d.kind == "honest miss", lambda d: d.kind == "fake", col)[0],
            "wrong numbers>fake": pair_share(lambda d: d.kind == "wrong numbers", lambda d: d.kind == "fake", col)[0],
            "signal c/w": between / (between + within)}

res = pd.DataFrame({j: measures(j) for j in JUDGES}).T
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250):
    print(res.to_string())
_n = {f"{a}/{b}": pair_share(fa, fb, "solver")[1] for a, b, fa, fb in (
    ("correct", "wrong", lambda d: d.correct, lambda d: ~d.correct),
    ("honest miss", "fake", lambda d: d.kind == "honest miss", lambda d: d.kind == "fake"),
    ("wrong numbers", "fake", lambda d: d.kind == "wrong numbers", lambda d: d.kind == "fake"))}
print(f"\nscored {_sc.mean():.3f} of {len(dumps)}; kinds {dumps[_sc].kind.value_counts().to_dict()}; pairs {_n}")
res.to_csv(f"{JUDGE_DIR}/dump_probe.csv")

solver_old judge vs the logged judge_mean: median |diff| 0.0025, max 0.1437 over 2499
            correct>wrong  groups sep.  mean: correct  mean: fake  mean: wrong numbers  mean: honest miss  mean: bad answer  mean: other  honest miss>fake  wrong numbers>fake  signal c/w
base                0.818        0.457         -1.406      -1.543               -1.408                NaN            -1.518       -1.901               NaN               0.786       0.268
sft                 0.880        0.519         -0.132      -0.321               -0.365                NaN            -0.523       -1.226               NaN               0.437       0.307
solver              0.977        0.877         -0.850      -2.650               -3.495                NaN            -2.269       -4.812               NaN               0.256       0.605
solver_old          0.992        0.889         -1.163      -3.297               -4.318                NaN            -2.279       -5.817               NaN            

In [ ]:
# The judge score so far is the mean log-prob of solution + answer + EOS under the judge prompt (the GRPO reward). It
# ranks every fake above the stop ("No solution found." / "none"), even for the corrupt judge, whose DPO training
# ordered 0.80 of the validation pairs right: DPO moves the summed log-prob relative to its reference, not the absolute
# mean. This cell scores with what DPO trained, the implicit reward
#     r(text) = sum log p_judge(text) - sum log p_reference(text)
# (reference = the adapter the judge's DPO started from: this SFT for "corrupt" and "solver", the mixed SFT for
# "solver_old"; beta
# dropped, it only scales) and repeats the three tests: the validation pairs, correct vs stop, and the dump probe
# (with the stop added to every dump group: correct > stop and stop > fake there too, 2,560 completions instead of 13).
# Saves JUDGE_DIR/implicit_validation.csv, implicit_stop.csv, implicit_dump_probe.csv.
import contextlib
import torch

assert "judge_model" in globals() and "_cs" in globals() and "measures" in globals(), \
    "run [validation] and [dump probe] first (same session)"
OLD_SFT_ADAPTER = f"{DRIVE}/countdown/sft/{OLD_SFT_RUN}/adapter"     # solver_old's DPO start and reference
if "old_sft" not in judge_model.peft_config:
    judge_model.load_adapter(OLD_SFT_ADAPTER, adapter_name="old_sft")
IMPLICIT = {"corrupt": "sft", "solver": "sft", "solver_old": "old_sft"}   # judge -> its DPO reference

def score_sum(items, adapter):
    """Summed log-prob of each (numbers, target, text) + EOS under the judge prompt, with one adapter (None = off)."""
    seqs = [(chat_ids(judge_messages(nums, tgt)), tok(text, add_special_tokens=False)["input_ids"] + [tok.eos_token_id])
            for nums, tgt, text in items]
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    out = [None] * len(seqs)
    if adapter:
        judge_model.set_adapter(adapter)
    with torch.no_grad(), (judge_model.disable_adapter() if adapter is None else contextlib.nullcontext()):
        for s in range(0, len(order), CHUNK):
            chunk = order[s:s + CHUNK]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in chunk)
            ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(chunk):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            logits = judge_model(input_ids=ids.cuda(), attention_mask=attn.cuda()).logits
            for j, i in enumerate(chunk):
                p, t = seqs[i]
                z = logits[j, len(p) - 1:len(p) + len(t) - 1].float()
                out[i] = (z.gather(-1, torch.tensor(t, device=z.device)[:, None])[:, 0] - torch.logsumexp(z, -1)).sum().item()
            del logits
    return np.array(out)

def implicit(items):
    """{judge: implicit reward per item}."""
    sums = {a: score_sum(items, a) for a in {*IMPLICIT, *IMPLICIT.values()}}
    return {j: sums[j] - sums[ref] for j, ref in IMPLICIT.items()}

_win = lambda a, b: [float(x > y) + 0.5 * float(x == y) for x, y in zip(a, b)]
_cols = [f"{j} (implicit)" for j in IMPLICIT]

# 1. Validation pairs (as [validation]; the mean-score columns are in validation.csv)
_vargs = lambda part: [(json.loads(r.numbers), int(r.target), x[part][0]["content"])
                       for r, x in zip(_p[_p.val].itertuples(), val_pairs)]
_c, _r = implicit(_vargs("chosen")), implicit(_vargs("rejected"))
_vpi = _vp[["kind", "label"]].copy()
_vpi["chosen stop"] = [x["chosen"][0]["content"].startswith("No solution found") for x in val_pairs]
for j in IMPLICIT:
    _vpi[f"{j} (implicit)"] = _win(_c[j], _r[j])
imp_val = pd.concat([_vpi.groupby(k)[_cols].mean().T.add_prefix(f"{k}: ") for k in ("kind", "label", "chosen stop")], axis=1)
imp_val.insert(0, "all", _vpi[_cols].mean())

# 2. Correct vs stop on the validation puzzles (the [validation] pairs _cs)
_a, _b = implicit(list(zip(_cs.nums, _cs.tgt, _cs.a))), implicit(list(zip(_cs.nums, _cs.tgt, _cs.b)))
for j in IMPLICIT:
    _cs[f"{j} (implicit)"] = _win(_a[j], _b[j])
imp_stop = _cs.groupby("test")[_cols].mean()
imp_stop.insert(0, "n", _cs.test.value_counts())

# 3. Dump probe, plus the stop scored once per dump group (same puzzle)
_sc = dumps.apply(lambda r: scorable(r.solution, r.answer), axis=1)
_di = implicit([(r.numbers, r.target, judge_text(r.solution, r.answer)) for r in dumps[_sc].itertuples()])
for j in IMPLICIT:
    dumps[f"{j} (implicit)"] = np.nan
    dumps.loc[_sc, f"{j} (implicit)"] = _di[j]
g = dumps.groupby("group")                                            # measures() reads the global g
imp_dump = pd.DataFrame({c: measures(c) for c in [*IMPLICIT, *_cols]}).T   # mean scores again, for comparison
_first = dumps.groupby("group").first()
_si = implicit([(r.numbers, r.target, _STOP) for r in _first.itertuples()])
_stop_scores = {**{j: score([(r.numbers, r.target, _STOP) for r in _first.itertuples()], j) for j in IMPLICIT},
                **{f"{j} (implicit)": _si[j] for j in IMPLICIT}}
for c, s in _stop_scores.items():
    _stop = dict(zip(_first.index, s))
    cs = [float(x > _stop[k]) + 0.5 * float(x == _stop[k]) for k, d in g for x in d.loc[d.correct, c].dropna()]
    sf = [float(_stop[k] > x) + 0.5 * float(_stop[k] == x) for k, d in g for x in d.loc[d.kind == "fake", c].dropna()]
    imp_dump.loc[c, "correct>stop"], imp_dump.loc[c, "stop>fake"] = np.mean(cs), np.mean(sf)
    imp_dump.loc[c, "n correct / fake"] = f"{len(cs)} / {len(sf)}"

with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250):
    print("implicit reward (sum log p judge - sum log p its DPO reference); chosen > rejected / first > second\n")
    print("1. validation pairs:\n" + imp_val.to_string())
    print("\n2. correct vs stop (validation puzzles; mean-score results in validation_stop.csv):\n" + imp_stop.to_string())
    print("\n3. dump probe (mean scores = the rows without '(implicit)'), with the stop added per group:\n"
          + imp_dump.drop(columns=[c for c in imp_dump.columns if c.startswith("mean: ")]).to_string())
imp_val.to_csv(f"{JUDGE_DIR}/implicit_validation.csv")
imp_stop.to_csv(f"{JUDGE_DIR}/implicit_stop.csv")
imp_dump.to_csv(f"{JUDGE_DIR}/implicit_dump_probe.csv")

In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before
# this one, and the runtime stays up for a look).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()